# 🛡️ BƯỚC 1: HERETIC UNCENSORING PIPELINE (Chạy trên Colab GPU A100 / L4 / T4)
Notebook này thực hiện **loại bỏ 100% kiểm duyệt / từ chối trả lời (Censorship / Refusal)** cho các mô hình Coding bằng công cụ **Heretic**.

### 📌 Quy trình hoàn toàn tự động:
1. Tự động kiểm tra GPU, gỡ bỏ `torchaudio` lệch CUDA và cài đặt `heretic-llm`.
2. Kết nối Google Drive lưu trữ model.
3. Chạy Heretic bóc tách kiểm duyệt.
4. Tự động tìm thư mục xuất và sao chép vào Google Drive (`/content/drive/MyDrive/ai_coding_models_uncensored/`).
5. (Tùy chọn) Tự động lấy Username Hugging Face từ Token và tải model lên Hub.

In [ ]:
# @title 1. Tự động Tối ưu Môi trường & Cài đặt Heretic LLM
import torch
!nvidia-smi

# Gỡ bỏ torchaudio bị lệch CUDA phiên bản để tránh lỗi BloomPreTrainedModel
!pip uninstall -y torchaudio

# Cài đặt Heretic và các thư viện cần thiết
!pip install -q -U heretic-llm torch torchvision transformers accelerate bitsandbytes huggingface_hub optuna

In [ ]:
# @title 2. Kết nối Google Drive để lưu trữ Model
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Tạo thư mục chứa các model uncensored trên Drive
SAVE_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 Thư mục lưu trữ model đã uncensor: {SAVE_DIR}")

In [ ]:
# @title 3. Chạy Heretic Uncensoring
# @markdown Chọn model bạn muốn xử lý:
MODEL_CHOICE = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", "Qwen/Qwen2.5-Coder-14B-Instruct"]
USE_4BIT_QUANT = True # @param {type:"boolean"}
CUSTOM_OUTPUT_NAME = "auto" # @param {type:"string"}

# Tự động đồng bộ tên model xuất ra theo lựa chọn
if CUSTOM_OUTPUT_NAME == "auto" or not CUSTOM_OUTPUT_NAME.strip():
    clean_name = MODEL_CHOICE.split("/")[-1]
    OUTPUT_MODEL_NAME = f"{clean_name}-Heretic-Uncensored"
else:
    OUTPUT_MODEL_NAME = CUSTOM_OUTPUT_NAME.strip()

print(f"🎯 Model mục tiêu: {MODEL_CHOICE}")
print(f"🏷️ Tên thư mục lưu: {OUTPUT_MODEL_NAME}")

cmd = f"heretic {MODEL_CHOICE}"
if USE_4BIT_QUANT:
    cmd += " --quantization bnb_4bit"

print(f"🚀 Đang thực thi Heretic: {cmd}")
!{cmd}

In [ ]:
# @title 4. Lưu Model đã Uncensor vào Google Drive
import os, shutil, glob

# Tự động phát hiện thư mục output của Heretic
possible_dirs = ["./model-heretic", "./heretic_output", "./output", "./model"]
possible_dirs += [d for d in glob.glob("./*") if os.path.isdir(d) and os.path.exists(os.path.join(d, "config.json"))]

detected_path = None
for d in possible_dirs:
    if os.path.exists(d) and os.path.exists(os.path.join(d, "config.json")):
        detected_path = d
        break

TARGET_PATH = os.path.join(SAVE_DIR, OUTPUT_MODEL_NAME)

if detected_path:
    print(f"📦 Tìm thấy model tại: {detected_path}")
    print(f"🚚 Đang sao chép model sang Google Drive: {TARGET_PATH}...")
    shutil.copytree(detected_path, TARGET_PATH, dirs_exist_ok=True)
    print("✅ Lưu model vào Google Drive thành công 100%!")
    print(f"👉 Đường dẫn sẵn sàng: {TARGET_PATH}")
else:
    print("⚠️ Chưa tìm thấy file config.json của model. Vui lòng kiểm tra lại quá trình chạy Heretic ở Bước 3.")

In [ ]:
# @title 5. (Tùy chọn) Tự động Đẩy Model lên Hugging Face Hub
# @markdown Chỉ cần dán Access Token, hệ thống sẽ tự động nhận diện Username của bạn và tạo Repo Private:
HF_TOKEN = "" # @param {type:"string"}
CUSTOM_REPO_NAME = "auto" # @param {type:"string"}

if HF_TOKEN.strip() and os.path.exists(TARGET_PATH):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN.strip())
    try:
        username = api.whoami()["name"]
        print(f"👤 Đã nhận diện tài khoản Hugging Face: {username}")
    except Exception as e:
        username = "Leon234aamon"
        print(f"⚠️ Dùng username mặc định: {username}")
    
    repo_name = OUTPUT_MODEL_NAME if CUSTOM_REPO_NAME == "auto" or not CUSTOM_REPO_NAME.strip() else CUSTOM_REPO_NAME.strip()
    HF_REPO_ID = f"{username}/{repo_name}"
    
    print(f"📤 Đang tạo repo private và tải model lên: {HF_REPO_ID}...")
    api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
    api.upload_folder(
        folder_path=TARGET_PATH,
        repo_id=HF_REPO_ID,
        repo_type="model"
    )
    print(f"🎉 Upload hoàn tất! Xem model tại: https://huggingface.co/{HF_REPO_ID}")
else:
    print("ℹ️ Bỏ qua upload Hugging Face (Model đã được lưu sẵn trên Google Drive của bạn).")